# 03 - Embedding

Generate and sanity-check embeddings with `BAAI/bge-m3` — the committed default,
not just a candidate. Scope stops at embeddings: Chroma indexing and actual
retrieval belong to `04_retrieval` (per CLAUDE.md's module boundaries). A full
model comparison (vs. `paraphrase-multilingual-MiniLM-L12-v2`) happens later, as
`03b_embedding_minilm`, once the full pipeline works end-to-end with this one.

## Step 1: Setup

Imports, load `data/processed/chunks.json` from `02_chunking`, and constants.

In [1]:
import json
from pathlib import Path

CHUNKS_PATH = Path("../data/processed/chunks.json")
EMBEDDINGS_PATH = Path("../data/processed/embeddings_bge_m3.npy")
CHUNK_IDS_PATH = Path("../data/processed/chunk_ids_bge_m3.json")

EMBEDDING_MODEL_NAME = "BAAI/bge-m3"

chunks = json.loads(CHUNKS_PATH.read_text(encoding="utf-8"))
len(chunks)

16

## Step 2: Load BGE-M3

Load `BAAI/bge-m3` via `sentence-transformers` (dense embeddings only for now —
BGE-M3's sparse/multi-vector modes aren't needed until hybrid retrieval in
`04_retrieval`).

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(EMBEDDING_MODEL_NAME)
model.get_sentence_embedding_dimension()

c:\Users\rames\Documents\GitHub\migrantBuddy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 37699.66it/s]
C:\Users\rames\AppData\Local\Temp\ipykernel_29564\3658372903.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  model.get_sentence_embedding_dimension()


1024

## Step 3: Embed chunks

Generate an embedding for every chunk's `text` (the breadcrumb-prefixed markdown
from `02_chunking`).

In [3]:
texts = [chunk["text"] for chunk in chunks]
embeddings = model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
embeddings.shape

Batches: 100%|██████████| 1/1 [00:14<00:00, 14.73s/it]


(16, 1024)

## Step 4: Sanity-check embeddings

Confirm expected dimensionality (1024) and no NaNs, then a basic quality check —
cosine similarity between a couple of related chunks should be noticeably higher
than between unrelated ones. Not a retrieval quality test (that needs real
queries, `04_retrieval`) — just confirming the embeddings aren't broken.

In [4]:
import numpy as np

assert embeddings.shape[1] == 1024, f"Unexpected embedding dimension: {embeddings.shape[1]}"
assert not np.isnan(embeddings).any(), "Found NaNs in embeddings"


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


# Find two chunks from the same document, and one from a different document --
# looked up by document_id, not guessed by index (index-based guessing breaks once
# the corpus has more than 2 documents with different chunk counts each).
first_doc_id = chunks[0]["document_id"]
same_doc_indices = [i for i, c in enumerate(chunks) if c["document_id"] == first_doc_id]
diff_doc_index = next(i for i, c in enumerate(chunks) if c["document_id"] != first_doc_id)

assert len(same_doc_indices) >= 2, "Need at least 2 chunks from the same document for this check"

idx_a, idx_b = same_doc_indices[0], same_doc_indices[1]
related_sim = cosine_similarity(embeddings[idx_a], embeddings[idx_b])
cross_doc_sim = cosine_similarity(embeddings[idx_a], embeddings[diff_doc_index])

print(f"{chunks[idx_a]['chunk_id']} vs {chunks[idx_b]['chunk_id']} (same doc): {related_sim:.3f}")
print(f"{chunks[idx_a]['chunk_id']} vs {chunks[diff_doc_index]['chunk_id']} (different doc): {cross_doc_sim:.3f}")

assert related_sim > cross_doc_sim, "Same-document chunks should be more similar than cross-document ones"
print("\nSanity checks passed.")

https-www-mom-gov-sg-employment-practices-salary-paying-salary::chunk-0 vs https-www-mom-gov-sg-employment-practices-salary-paying-salary::chunk-1 (same doc): 0.831
https-www-mom-gov-sg-employment-practices-salary-paying-salary::chunk-0 vs https-www-mom-gov-sg-employment-practices-hours-of-work-overtime-and-rest-days::chunk-0 (different doc): 0.686

Sanity checks passed.


## Step 5: Save embeddings

Save the embeddings (aligned to `chunk_id`) to `data/processed/embeddings_bge_m3.npy`
plus a chunk-id order file, so `04_retrieval` can load them without recomputing.

In [5]:
np.save(EMBEDDINGS_PATH, embeddings)
CHUNK_IDS_PATH.write_text(
    json.dumps([chunk["chunk_id"] for chunk in chunks], indent=2),
    encoding="utf-8",
)

EMBEDDINGS_PATH, CHUNK_IDS_PATH

(WindowsPath('../data/processed/embeddings_bge_m3.npy'),
 WindowsPath('../data/processed/chunk_ids_bge_m3.json'))